# 1 · SFT (Supervised Fine-Tuning)
The model learns from **examples with their correct answers**, like a student studying solved exercises.

## When is fine-tuning worth it?
| Prompting is enough | Fine-tuning helps |
|---|---|
| general knowledge, a few clear rules | many house rules, codes or formats the model has never seen |
| a large model is available | a small, fast, local model must do the job |
| no examples exist | many past examples exist |

## The task: a shop's search box
Customers write freely. The shop's system needs its own command:

`black sneakers, under 50 dollars, newest first` → `FIND shoes color=BK max=50 sort=new`

The model cannot know this format or the color codes. We try two ways:
- **A. Prompting only:** every rule written into a long prompt.
- **B. Fine-tuning:** many examples, then a short prompt.

## Key words
| Word | Meaning |
|---|---|
| **token** | a small piece of text, like `FIND`, `BK` or `sneak`. The model reads and writes tokens. |
| **confidence** | how sure the model is that a token comes next, from 0% to 100% |
| **training** | many small adjustments so the correct answers get more confidence |
| **loss** | one number for *how wrong* the model is. Training makes it go down. |
| **LoRA** | a small add-on (about 1% of the model) that is trained instead of the whole model |
| **epoch** | one pass through all training examples |

## Step 1 · Load the model
A small open model (0.8 billion parameters). It downloads once.

In [ ]:
import random, warnings, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, logging

warnings.filterwarnings('ignore')             # keep the output readable
logging.set_verbosity_error()

MODEL = 'Qwen/Qwen3.5-0.8B'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cuda')

## Step 2 · Create 300 examples
The house rules are small word lists. Code mixes them into requests, each with its correct command.
260 examples to learn from, 40 to test (never used for training).

In [ ]:
ITEMS = {'shirt': ['shirt', 't-shirt', 'tee'],
         'shoes': ['shoes', 'sneakers', 'trainers'],
         'jacket': ['jacket', 'coat'],
         'hoodie': ['hoodie', 'sweatshirt']}
COLORS = {'red': 'RD', 'blue': 'BL', 'black': 'BK', 'white': 'WH'}
PRICES = ['under {} dollars', 'less than ${}', 'below {}', 'max {} bucks']
SORTS = {'price_asc': ['cheapest first', 'lowest price first'],
         'new': ['newest first', 'latest arrivals']}
STARTS = ['', 'show me ', 'I want ', 'looking for ']

def make_example():
    item = random.choice(list(ITEMS))
    color = random.choice(list(COLORS))
    request = random.choice(STARTS) + color + ' ' + random.choice(ITEMS[item])
    command = f'FIND {item} color={COLORS[color]}'
    if random.random() < 0.6:                       # sometimes a price limit
        price = random.choice([20, 30, 50, 80, 100])
        request += ', ' + random.choice(PRICES).format(price)
        command += f' max={price}'
    if random.random() < 0.5:                       # sometimes a sort order
        sort = random.choice(list(SORTS))
        request += ', ' + random.choice(SORTS[sort])
        command += f' sort={sort}'
    return request, command

random.seed(0)
examples = {}
while len(examples) < 300:
    request, command = make_example()
    examples[request] = command                     # each request only once
examples = list(examples.items())
TRAIN, TEST = examples[:260], examples[260:]

for request, command in TRAIN[:6]:
    print(f'{request:50} -> {command}')

## Step 3 · A. Prompting only
Every rule is written into the prompt, plus 3 examples. No training.

**Look for:** the mistakes, and how long the prompt is.

In [ ]:
RULES = '''Convert a shopping request into a search command.
Format: FIND <item> color=<code> max=<price> sort=<order>   (max and sort only if asked)
Items: shirt = shirt, t-shirt, tee | shoes = shoes, sneakers, trainers | jacket = jacket, coat | hoodie = hoodie, sweatshirt
Colors: red=RD, blue=BL, black=BK, white=WH
Sort: price_asc = cheapest first, lowest price first | new = newest first, latest arrivals
Examples:
red tee, below 30 -> FIND shirt color=RD max=30
I want white coat, newest first -> FIND jacket color=WH sort=new
looking for black trainers -> FIND shoes color=BK
Reply with only the command.'''

def long_prompt(request):        # A: every rule written out
    return f'{RULES}\n\nRequest: {request}'

def short_prompt(request):       # B: the rules are learned from examples
    return f'Search command for: {request}'

def ask(model, prompt):
    x = tok.apply_chat_template([{'role': 'user', 'content': prompt}], add_generation_prompt=True,
                                return_tensors='pt', return_dict=True).to('cuda')
    y = model.generate(**x, max_new_tokens=30, do_sample=False)
    return tok.decode(y[0][x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def evaluate(model, make_prompt, show=3):
    model.eval()
    mistakes = []
    for request, command in TEST:
        answer = ask(model, make_prompt(request))
        if answer != command:
            mistakes.append((request, command, answer))
    for request, command, answer in mistakes[:show]:
        print(f'  {request}\n     expected: {command}\n     got:      {answer!r}')
    accuracy = 1 - len(mistakes) / len(TEST)
    print(f'accuracy: {accuracy:.0%}  ({len(TEST) - len(mistakes)} of {len(TEST)} correct)')
    return accuracy

long_tokens = len(tok(long_prompt(TEST[0][0])).input_ids)
short_tokens = len(tok(short_prompt(TEST[0][0])).input_ids)
print(f'long prompt: {long_tokens} tokens per request\n')
prompt_only = evaluate(model, long_prompt)

## Step 4 · B. Fine-tuning: how sure is the model before training?
With the short prompt, the model has never seen this format.
For each token of the correct command: how sure is the model that this token comes next?

**Look for:** low confidence on `FIND` and on the color code. Training must raise it.

In [ ]:
def show_confidence(model, request, command):
    model.eval()
    q = tok.apply_chat_template([{'role': 'user', 'content': short_prompt(request)}],
                                add_generation_prompt=True, tokenize=False)
    start = len(tok(q).input_ids)                   # where the command begins
    ids = tok(q + command, return_tensors='pt').input_ids.to('cuda')
    with torch.no_grad():
        confidence = torch.softmax(model(input_ids=ids).logits[0].float(), dim=-1)   # scores -> 0-100%
    for pos in range(start, ids.shape[1]):
        token = ids[0, pos]
        print(f'{tok.decode(token)!r:12} {confidence[pos - 1, token].item():6.1%}')   # guessed one step earlier

request, command = TEST[0]
print(request, '->', command, '\n')
show_confidence(model, request, command)

## Step 5 · Prepare the training
Each example becomes one row: the short prompt + the correct command.

**What the trainer really sees:** tokens. Prompt tokens are marked `-` (only context).
Command tokens are marked `learned`: only these are trained.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

data = Dataset.from_list([
    {'prompt': [{'role': 'user', 'content': short_prompt(r)}],
     'completion': [{'role': 'assistant', 'content': c}]} for r, c in TRAIN])

trainer = SFTTrainer(
    model=model,
    processing_class=tok,
    train_dataset=data,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules='all-linear'),   # the small add-on
    args=SFTConfig(num_train_epochs=1, learning_rate=2e-4, logging_steps=10),     # 1 epoch per call
)
trainer.model.print_trainable_parameters()

row = trainer.train_dataset[0]
for token_id, label in zip(row['input_ids'], row['labels']):
    print(f'{tok.decode(token_id)!r:22}', 'learned' if label != -100 else '-')

## Step 6 · Train, one epoch at a time
Each step: take 8 examples → measure the loss → adjust the add-on a little.
After each epoch, the 40 test requests are checked.

**Look for:** the `loss` going down, the accuracy going up, and the mistakes disappearing.

In [ ]:
print('before training:')
fine_tuned = evaluate(trainer.model, short_prompt)

for epoch in [1, 2, 3]:
    trainer.train()                                 # one pass over the 260 examples
    print(f'\nafter epoch {epoch}:')
    fine_tuned = evaluate(trainer.model, short_prompt)

## Step 7 · Did it work?
The same test example as Step 4, then A and B side by side.
It worked if the fine-tuned model clearly beats prompting only.

In [ ]:
show_confidence(trainer.model, request, command)

print(f'\nA. prompting only: {prompt_only:.0%} correct, {long_tokens} prompt tokens')
print(f'B. fine-tuned:     {fine_tuned:.0%} correct, {short_tokens} prompt tokens')

if prompt_only >= 0.95:
    print('\nPrompting alone was already enough: no training needed.')
elif fine_tuned - prompt_only >= 0.10:
    print('\nIt worked: training clearly beat prompting only.')
else:
    print('\nNo clear gain: try more epochs or more examples.')

## Summary
- **Needs:** many examples with correct answers (here 260)
- **Learns:** the house rules, from examples, without writing them down
- **Signs it works:** the loss goes down, the test accuracy goes up
- **Result:** a small model follows a new format with a short prompt